In [1]:
from sklearn.datasets import make_classification
import torch


In [2]:
torch.cuda.get_device_name(0)

'NVIDIA GeForce RTX 2050'

In [9]:
X,y=make_classification(
    n_samples=10,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    n_classes=2,
    random_state=42
)

In [10]:
X

array([[ 1.06833894, -0.97007347],
       [-1.14021544, -0.83879234],
       [-2.8953973 ,  1.97686236],
       [-0.72063436, -0.96059253],
       [-1.96287438, -0.99225135],
       [-0.9382051 , -0.54304815],
       [ 1.72725924, -1.18582677],
       [ 1.77736657,  1.51157598],
       [ 1.89969252,  0.83444483],
       [-0.58723065, -1.97171753]])

In [11]:
y

array([1, 0, 0, 0, 0, 1, 1, 1, 1, 0])

In [12]:
X=torch.tensor(X, dtype=torch.float32)
y=torch.tensor(y, dtype=torch.long)

In [13]:
X,y

(tensor([[ 1.0683, -0.9701],
         [-1.1402, -0.8388],
         [-2.8954,  1.9769],
         [-0.7206, -0.9606],
         [-1.9629, -0.9923],
         [-0.9382, -0.5430],
         [ 1.7273, -1.1858],
         [ 1.7774,  1.5116],
         [ 1.8997,  0.8344],
         [-0.5872, -1.9717]]),
 tensor([1, 0, 0, 0, 0, 1, 1, 1, 1, 0]))

In [14]:
from torch.utils.data import Dataset, DataLoader

In [15]:
## Defining the custom Dataset class

class customDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels

    def __len__(self):
        return self.features.shape[0]

    def __getitem__(self, index):
        return self.features[index], self.labels[index]

dataset=customDataset(X,y)


In [16]:
len(dataset)

10

In [17]:
dataset[0]

(tensor([ 1.0683, -0.9701]), tensor(1))

In [18]:
dataset[1]

(tensor([-1.1402, -0.8388]), tensor(0))

In [19]:
DataLoader?

Init signature:
DataLoader(
    dataset: 'Dataset[_T_co]',
    batch_size: 'int | None' = 1,
    shuffle: 'bool | None' = None,
    sampler: 'Sampler | Iterable | None' = None,
    batch_sampler: 'Sampler[list] | Iterable[list] | None' = None,
    num_workers: 'int' = 0,
    collate_fn: '_collate_fn_t | None' = None,
    pin_memory: 'bool' = False,
    drop_last: 'bool' = False,
    timeout: 'float' = 0,
    worker_init_fn: '_worker_init_fn_t | None' = None,
    multiprocessing_context=None,
    generator=None,
    *,
    prefetch_factor: 'int | None' = None,
    persistent_workers: 'bool' = False,
    pin_memory_device: 'str' = '',
    in_order: 'bool' = True,
) -> 'None'
Docstring:     
Data loader combines a dataset and a sampler, and provides an iterable over the given dataset.

The :class:`~torch.utils.data.DataLoader` supports both map-style and
iterable-style datasets with single- or multi-process loading, customizing
loading order and optional automatic batching (collation) and

In [20]:
## defining the loader
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

In [21]:
for batch_features , batch_labels in dataloader:
    print(batch_features)
    print(batch_labels)
    print("-"*10)

tensor([[-0.9382, -0.5430],
        [-1.1402, -0.8388]])
tensor([1, 0])
----------
tensor([[ 1.0683, -0.9701],
        [ 1.7774,  1.5116]])
tensor([1, 1])
----------
tensor([[ 1.8997,  0.8344],
        [-0.5872, -1.9717]])
tensor([1, 0])
----------
tensor([[-1.9629, -0.9923],
        [-2.8954,  1.9769]])
tensor([0, 0])
----------
tensor([[ 1.7273, -1.1858],
        [-0.7206, -0.9606]])
tensor([1, 0])
----------


## entire working on the BrestCancer Dataset

In [22]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

In [25]:
df = pd.read_csv("https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/master/data.csv")


In [26]:
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [27]:
df.drop(columns=['id', 'Unnamed: 32'], inplace=True)

In [28]:
df

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.30010,0.14710,0.2419,...,25.380,17.33,184.60,2019.0,0.16220,0.66560,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.08690,0.07017,0.1812,...,24.990,23.41,158.80,1956.0,0.12380,0.18660,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.19740,0.12790,0.2069,...,23.570,25.53,152.50,1709.0,0.14440,0.42450,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.24140,0.10520,0.2597,...,14.910,26.50,98.87,567.7,0.20980,0.86630,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.19800,0.10430,0.1809,...,22.540,16.67,152.20,1575.0,0.13740,0.20500,0.4000,0.1625,0.2364,0.07678
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
564,M,21.56,22.39,142.00,1479.0,0.11100,0.11590,0.24390,0.13890,0.1726,...,25.450,26.40,166.10,2027.0,0.14100,0.21130,0.4107,0.2216,0.2060,0.07115
565,M,20.13,28.25,131.20,1261.0,0.09780,0.10340,0.14400,0.09791,0.1752,...,23.690,38.25,155.00,1731.0,0.11660,0.19220,0.3215,0.1628,0.2572,0.06637
566,M,16.60,28.08,108.30,858.1,0.08455,0.10230,0.09251,0.05302,0.1590,...,18.980,34.12,126.70,1124.0,0.11390,0.30940,0.3403,0.1418,0.2218,0.07820
567,M,20.60,29.33,140.10,1265.0,0.11780,0.27700,0.35140,0.15200,0.2397,...,25.740,39.42,184.60,1821.0,0.16500,0.86810,0.9387,0.2650,0.4087,0.12400


In [29]:

X_train, X_test, y_train, y_test = train_test_split(df.iloc[:,1:], df.iloc[:,0], test_size=0.2)
print(X_train.shape)

scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

encoder=LabelEncoder()
y_train=encoder.fit_transform(y_train)
y_test=encoder.transform(y_test)

X_train_tensor = torch.from_numpy(X_train.astype(np.float32))
X_test_tensor = torch.from_numpy(X_test.astype(np.float32))
y_train_tensor = torch.from_numpy(y_train.astype(np.float32))
y_test_tensor = torch.from_numpy(y_test.astype(np.float32))

print(X_train_tensor.shape)
print(y_train_tensor.shape)


(455, 30)
torch.Size([455, 30])
torch.Size([455])


In [30]:
from torch.utils.data import Dataset, DataLoader

class customDataset(Dataset):
    def __init__(self, features, labels):
        self.features=features
        self.labels=labels

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

In [31]:
train_dataset=customDataset(X_train_tensor, y_train_tensor)
test_dataset=customDataset(X_test_tensor, y_test_tensor)

In [32]:
train_dataset[10]

(tensor([-0.5346, -0.9993, -0.5877, -0.5460, -0.6723, -1.1096, -0.8446, -0.7388,
         -0.8827, -0.7738, -0.0731, -0.5535, -0.1268, -0.2209,  0.2674, -0.9558,
         -0.6382, -0.2300,  1.1074, -0.8988, -0.6602, -1.4318, -0.7052, -0.6212,
         -1.2113, -1.1828, -1.0692, -1.0166, -0.9545, -1.3464]),
 tensor(0.))

In [33]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)


In [34]:
## Defining the model 
import torch.nn as nn
class mySimpleNN(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        self.linear = nn.Linear(num_features, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, features):
        out=self.linear(features)
        out=self.sigmoid(out)
        return out



In [35]:
## important parameters
learning_rate=0.1
epochs=25

## create the model 
model=mySimpleNN(X_train_tensor.shape[1])

# define optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

## define loss function
loss_function=nn.BCELoss()

### Training Pipeline

What .view() Does
Reshapes a tensor: Changes how data is interpreted, not the data itself.

Shares memory: The reshaped tensor is just a different “view” of the same storage.

Requires contiguity: The tensor must be stored contiguously in memory; otherwise, you need .contiguous() first.

Fast and memory-efficient: No data copy happens.

In [36]:
for epoch in range(epochs):
    for batch_features, batch_labels in train_loader:

        # forward pass
        y_pred = model(batch_features)

        # loss calculate
        loss= loss_function(y_pred, batch_labels.view(-1,1))

        # clear gradients
        optimizer.zero_grad()
        #backward pass
        loss.backward()
        #parameter update
        optimizer.step()

    print(f"Epoch No.: {epoch+1} : Loss: {loss.item()}")


Epoch No.: 1 : Loss: 0.19492726027965546
Epoch No.: 2 : Loss: 0.16777552664279938
Epoch No.: 3 : Loss: 0.10382292419672012
Epoch No.: 4 : Loss: 0.16111502051353455
Epoch No.: 5 : Loss: 0.16375084221363068
Epoch No.: 6 : Loss: 0.31667613983154297
Epoch No.: 7 : Loss: 0.1400635540485382
Epoch No.: 8 : Loss: 0.04016399383544922
Epoch No.: 9 : Loss: 0.08261817693710327
Epoch No.: 10 : Loss: 0.17424069344997406
Epoch No.: 11 : Loss: 0.06283508986234665
Epoch No.: 12 : Loss: 0.06468372792005539
Epoch No.: 13 : Loss: 0.03697408363223076
Epoch No.: 14 : Loss: 0.07764606177806854
Epoch No.: 15 : Loss: 0.012096481397747993
Epoch No.: 16 : Loss: 0.0077314781956374645
Epoch No.: 17 : Loss: 0.0805429071187973
Epoch No.: 18 : Loss: 0.0673513412475586
Epoch No.: 19 : Loss: 0.01982194557785988
Epoch No.: 20 : Loss: 0.11774381250143051
Epoch No.: 21 : Loss: 0.0540127195417881
Epoch No.: 22 : Loss: 0.0027243229560554028
Epoch No.: 23 : Loss: 0.022767210379242897
Epoch No.: 24 : Loss: 0.05652854591608047